Prepare CMIP Wind Speed for Scenario 4

In [1]:
from pathlib import Path
import numpy as np
import torch
import xarray as xr

project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "experiments" / "02_climate").is_dir()
)

experiment_dir = project_root / "experiments" / "02_climate"
raw_dir = project_root / "datasets" / "climate" / "raw"

# Reuse spatial grid positions from Scenario 2
locations_file = (
    experiment_dir / "data" / "pressure_nearest_5sites.pt"
)

locations = torch.load(
    locations_file, map_location="cpu", weights_only=True
)

rows = locations["grid_rows"]
cols = locations["grid_cols"]
location_indices = locations["location_indices"]

# Read 2920 timestamps from original wind NetCDF
wind_path = raw_dir / "Klima_Daten_10m_3h_2020_RCP26.nc"

with xr.open_dataset(wind_path) as ds:
    wind_selected = ds["sfcWind"].isel(
        time=slice(0, 2920),
        rlat=xr.DataArray(rows, dims="site"),
        rlon=xr.DataArray(cols, dims="site")
    )

    wind_raw = (
        wind_selected
        .transpose("time", "site")
        .values.astype(np.float32)
    )

assert wind_raw.shape == (2920, 5)
assert np.isfinite(wind_raw).all()

# Chronological 80/20 split
split_index = int(0.8 * len(wind_raw))

output_path = (
    experiment_dir / "data" / "wind_2920_5sites.pt"
)

torch.save({
    "wind_raw": torch.from_numpy(wind_raw),
    "location_indices": location_indices,
    "grid_rows": rows,
    "grid_cols": cols,
    "train_split_index": split_index,
    "source": wind_path.name
}, output_path)

print("=== SCENARIO 4: CMIP WIND SPEED ===")
print("Raw:", wind_raw.shape)
print("Train:", wind_raw[:split_index].shape)
print("Test:", wind_raw[split_index:].shape)
print("Locations:", location_indices)
print("Saved:", output_path)


c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


Train/Test Split, Scaling and Sliding Windows

In [3]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import torch

# 1. Chronological train/test split (80/20)
train_raw = wind_raw[:split_index]
test_raw = wind_raw[split_index:]

# 2. Fit scaler on train only
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_raw)
test_scaled = scaler.transform(test_raw)

# 3. Sliding-window function
def build_windows(data, seq_len, horizon):
    X, Y = [], []

    for i in range(len(data) - seq_len - horizon):
        X.append(data[i:i + seq_len])
        Y.append(
            data[i + seq_len:i + seq_len + horizon]
            .T.reshape(-1)
        )

    x_in = torch.tensor(np.stack(X), dtype=torch.float32)
    x_out = torch.tensor(np.stack(Y), dtype=torch.float32)

    return x_in, x_out

# 4. Scenario 4 configuration
seq_len = 120
patch_lengths = [40, 50, 60, 70, 80, 90, 100]
horizons = [2, 4, 6, 8, 10, 12, 14, 16]

d_model = 48
epochs = 4000
learning_rate = 3e-4
lyap_weight = 0.1
seed = 7

# 5. Example: forecast horizon = 10
example_horizon = 10

x_train, y_train = build_windows(
    train_scaled, seq_len, example_horizon
)

x_test, y_test = build_windows(
    test_scaled, seq_len, example_horizon
)

# 6. Validation
assert x_train.shape == (2206, 120, 5)
assert y_train.shape == (2206, 50)
assert x_test.shape == (454, 120, 5)
assert y_test.shape == (454, 50)

print("=== SCENARIO 4: TRAIN/TEST DATASET ===")
print("Raw:", wind_raw.shape)
print("Train raw:", train_raw.shape)
print("Test raw:", test_raw.shape)

print("\nAfter sliding windows (H=10):")
print("X_train:", x_train.shape)
print("Y_train:", y_train.shape)
print("X_test:", x_test.shape)
print("Y_test:", y_test.shape)

print("\nScaling:")
print("Train range:", train_scaled.min(), train_scaled.max())
print("Test range:", test_scaled.min(), test_scaled.max())

print("\nGrid Search:")
print("Input sequence length:", seq_len)
print("Patch lengths:", patch_lengths)
print("Horizons:", horizons)
print("d_model:", d_model)
print("Epochs:", epochs)
print("Configurations:", len(patch_lengths) * len(horizons))


=== SCENARIO 4: TRAIN/TEST DATASET ===
Raw: (2920, 5)
Train raw: (2336, 5)
Test raw: (584, 5)

After sliding windows (H=10):
X_train: torch.Size([2206, 120, 5])
Y_train: torch.Size([2206, 50])
X_test: torch.Size([454, 120, 5])
Y_test: torch.Size([454, 50])

Scaling:
Train range: 0.0 1.0
Test range: -0.002704557 1.1441792

Grid Search:
Input sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
d_model: 48
Epochs: 4000
Configurations: 56


Save Scenario 4 Dataset for Kaggle

In [4]:
from pathlib import Path
import torch
import numpy as np

# 1. Output path
output_dir = experiment_dir / "data"
output_dir.mkdir(parents=True, exist_ok=True)

save_path = output_dir / "scenario4_wind_train_test.pt"

# 2. Save normalized train/test data and experiment settings
checkpoint = {
    "train_scaled": torch.tensor(train_scaled, dtype=torch.float32),
    "test_scaled": torch.tensor(test_scaled, dtype=torch.float32),
    "train_raw": torch.tensor(train_raw, dtype=torch.float32),
    "test_raw": torch.tensor(test_raw, dtype=torch.float32),

    "location_indices": list(location_indices),
    "scaler_min": torch.tensor(scaler.data_min_, dtype=torch.float32),
    "scaler_max": torch.tensor(scaler.data_max_, dtype=torch.float32),

    "seq_len": seq_len,
    "patch_lengths": patch_lengths,
    "horizons": horizons,
    "d_model": d_model,
    "epochs": epochs,
    "learning_rate": learning_rate,
    "lyap_weight": lyap_weight,
    "seed": seed,
    "train_fraction": 0.8,
}

torch.save(checkpoint, save_path)

# 3. Verify saved file
saved = torch.load(
    save_path,
    map_location="cpu",
    weights_only=True
)

assert saved["train_scaled"].shape == (2336, 5)
assert saved["test_scaled"].shape == (584, 5)
assert saved["seq_len"] == 120
assert saved["epochs"] == 4000

print("=== SCENARIO 4 DATASET SAVED ===")
print("Train:", saved["train_scaled"].shape)
print("Test:", saved["test_scaled"].shape)
print("Input sequence length:", saved["seq_len"])
print("Patch lengths:", saved["patch_lengths"])
print("Horizons:", saved["horizons"])
print("Epochs:", saved["epochs"])
print("Total training runs:",
      len(patch_lengths) * len(horizons) * 4)
print("File size (MB):", round(save_path.stat().st_size / 1024**2, 3))
print("Saved:", save_path)


=== SCENARIO 4 DATASET SAVED ===
Train: torch.Size([2336, 5])
Test: torch.Size([584, 5])
Input sequence length: 120
Patch lengths: [40, 50, 60, 70, 80, 90, 100]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]
Epochs: 4000
Total training runs: 224
File size (MB): 0.114
Saved: c:\Users\Win 11 New\Desktop\pa gk\experiments\02_climate\data\scenario4_wind_train_test.pt
